# Source gallery: nine light-source families, one pipeline

The whole `highuvlith` pipeline is generic over the `LithographySource`
trait, so every source - from a 157 nm VUV excimer down to a 6.7 nm BEUV
plasma - runs through the **same** aerial-imaging code. This notebook builds
all nine families from their factory constructors, tabulates what each one
reports, and images the **same normalized mask** through every one.

> **Honesty first.** These families sit at very different maturity levels.
> Some are validated against analytical physics; others are runnable research
> projections. The status table at the end (drawn from
> `docs/capability-matrix.md`) says which is which - read it before quoting a
> number.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import highuvlith as huv

%matplotlib inline

## Build every family from its factory

`SourceConfig` exposes a static factory per source. The eleven presets below
span **nine physical families** (VUV and LPP each appear twice, at two
wavelengths):

| Family | Factories |
|--------|-----------|
| VUV excimer | `f2_laser`, `ar2_laser` |
| LPA-FEL | `lpa_fel_bella_25nm` |
| LPP | `lpp_sn_13nm5`, `lpp_gd_6nm7` |
| Synchrotron | `synchrotron_undulator` |
| HHG | `hhg_ne_13nm5` |
| XFEL | `xfel_flash_13nm5` |
| Inverse Compton | `ics` |
| SSMB | `ssmb_euv_13nm5` |
| Entangled (NOON) | `entangled_noon` |

In [ ]:
factories = [
    "f2_laser", "ar2_laser", "lpa_fel_bella_25nm",
    "lpp_sn_13nm5", "lpp_gd_6nm7", "synchrotron_undulator",
    "hhg_ne_13nm5", "xfel_flash_13nm5", "ics",
    "ssmb_euv_13nm5", "entangled_noon",
]
sources = [(name, getattr(huv.SourceConfig, name)()) for name in factories]
print(f"built {len(sources)} source presets")

## Comparison table

Each source reports the physical quantities the pipeline actually reads:
kind tag, wavelength, time-averaged power, shot-to-shot RMS dose jitter (which
feeds the stochastic LER model), and the transverse coherence fraction. Fields
a given source does not define come back as `None`.

In [ ]:
def fmt(v, unit="", nd=3):
    return "     -   " if v is None else f"{v:.{nd}g}{unit}"

header = f"{'factory':<24}{'kind':<12}{'lambda/nm':>10}{'P_avg/W':>12}{'rms':>8}{'coh':>7}"
print(header)
print("-" * len(header))
for name, s in sources:
    print(
        f"{name:<24}{s.kind:<12}{s.wavelength_nm:>10.3f}"
        f"{fmt(s.average_power_w, nd=3):>12}"
        f"{fmt(s.shot_to_shot_rms, nd=2):>8}"
        f"{fmt(s.transverse_coherence_fraction, nd=2):>7}"
    )

## The same mask through every source

To compare imaging on equal footing we scale the mask to each wavelength: a
**3$\lambda$ line on a 6$\lambda$ pitch** (constant in $k_1$ units). The two
VUV lasers and the entangled source use a refractive-class NA of **0.75**; the
short-wavelength reflective/zone-plate families use **NA 0.33**. We read the
resulting aerial image contrast for each.

In [ ]:
def contrast_for(src, na, grid_size=64, max_kernels=6):
    wl = src.wavelength_nm
    cd, pitch = 3 * wl, 6 * wl
    optics = huv.OpticsConfig(numerical_aperture=na)
    mask = huv.MaskConfig.line_space(cd_nm=cd, pitch_nm=pitch)
    grid = huv.GridConfig(size=grid_size, pixel_nm=max(wl / 4, 1.0))
    engine = huv.SimulationEngine(src, optics, mask, grid=grid, max_kernels=max_kernels)
    return engine.compute_aerial_image(focus_nm=0.0).image_contrast()

results = []
for name, s in sources:
    na = 0.75 if s.kind in ("vuv", "entangled") else 0.33
    results.append((name, s.kind, s.wavelength_nm, na, contrast_for(s, na)))
    print(f"{name:<24} contrast = {results[-1][-1]:.3f}")

In [ ]:
names = [r[0] for r in results]
contrasts = [r[4] for r in results]
colors = ["#c1272d" if r[3] == 0.75 else "#1f6feb" for r in results]

plt.figure(figsize=(9, 4.5))
plt.bar(range(len(names)), contrasts, color=colors)
plt.xticks(range(len(names)), names, rotation=60, ha="right")
plt.ylabel("aerial image contrast")
plt.ylim(0, 1.0)
plt.title("3lambda:6lambda line/space contrast across nine source families")
handles = [
    plt.Rectangle((0, 0), 1, 1, color="#c1272d"),
    plt.Rectangle((0, 0), 1, 1, color="#1f6feb"),
]
plt.legend(handles, ["NA 0.75 (VUV / entangled)", "NA 0.33 (EUV/BEUV)"])
plt.tight_layout()

## Honest per-family status

From `docs/capability-matrix.md` (the single source of truth). Legend:
**✅ implemented & physics-validated** · **🔶 simplified / documented
approximations** · **🧪 theoretical / research projection**.

| Family | Status | What is (and isn't) modeled |
|--------|--------|-----------------------------|
| VUV excimer (F$_2$ 157.63 nm, Ar$_2$ 126 nm) | ✅ | Line shape and pupil fill live; pulse energy $\times$ rep-rate $\to$ average power. |
| LPA-FEL (20-30 nm, BELLA-class) | 🔶 | Spectral/pupil model live; pulse jitter feeds LER; electron energy documentary; default pupil still conventional. |
| LPP (Sn 13.5 nm / Gd 6.7 nm) | ✅ | In-band Gaussian spectrum; power chain (drive $\times$ CE $\times$ transport) live; plasma dynamics/debris out of scope. |
| Synchrotron (undulator) | ✅ | Wavelength **derived** from machine parameters; Kostroun spectrum; Gaussian line (not sinc$^2$). |
| HHG table-top EUV | ✅/🔶 | Comb spacing + cutoff law exact; plateau rolloff empirical; full-comb mode is spectral bookkeeping, not honest imaging. |
| XFEL (SASE / self-seeded) | ✅ | SASE bandwidth $2\rho$; Gamma statistics $\to$ live LER dose jitter; spike realizations research-only. |
| Inverse Compton scattering | 🧪 | Kinematics exact; flux at litho dose rates undemonstrated. |
| SSMB storage ring | 🧪 | Harmonic-consistency check live; kW average power is a projection. |
| Entangled-photon (NOON) | 🧪 | Reports the physical $\lambda$; quantum $\lambda/2N$ sharpening is applied explicitly, not here. |

The aerial contrasts above look similar because a well-resolved 3$\lambda$:6$\lambda$
pattern images cleanly through any of them - the **real** differences live in
the table two cells up (power, coherence, jitter) and in the maturity column
here.